# 04 - Silver: CIIU (Catálogo de Actividades Económicas)

Trabajo Práctico 2 - Transformación Bronze → Silver con reglas de calidad (DQL)

## Objetivo

Tipar y validar `Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api` (499 filas). Las filas que pasan todas las reglas de calidad van a `Datos_Empresas.silver.actividades_ciiu`; las que incumplen al menos una van a `Datos_Empresas.dql.rechazados_ciiu`.

## Transformaciones de tipado

* `id`, `status`, `version`: texto → `INT`.
* `code`, `description`: se recortan espacios (`trim`).

## Reglas de calidad (rechazo)

| # | Regla | Justificación |
|---|---|---|
| 1 | `code` vacío | Es la clave del catálogo — sin ella no se puede enlazar con `cod_ciiu_act_econ_pri` de RUES en Gold. |
| 2 | `code` duplicado | Regla de unicidad: debe existir una única descripción por código CIIU; un duplicado generaría filas repetidas al hacer `JOIN` con RUES en Gold. |

In [ ]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABLA_BRONZE = "Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api"
TABLA_SILVER = "Datos_Empresas.silver.actividades_ciiu"
TABLA_DQL = "Datos_Empresas.dql.rechazados_ciiu"

df_bronze = spark.table(TABLA_BRONZE)
print(f"Bronze: {df_bronze.count():,} filas")

---

## Paso 1.5: Diagnóstico previo — evidencia detrás de cada regla

Resultado real contra Bronze (499 filas): Regla 1 (code vacío) = 0, Regla 2 (code duplicado) = 0. El catálogo CIIU está completamente limpio; las reglas quedan como salvaguarda para cargas futuras, no porque haya rechazos hoy.

In [ ]:
%sql
-- Evidencia Regla 1: code vacío
SELECT COUNT(*) AS filas_afectadas
FROM Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api
WHERE TRIM(code) IS NULL OR TRIM(code) = '';

In [ ]:
%sql
-- Evidencia Regla 2: code duplicado
SELECT code, COUNT(*) AS repeticiones
FROM Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api
GROUP BY code
HAVING COUNT(*) > 1
ORDER BY repeticiones DESC;

---

## Paso 1: Tipado y reglas de calidad

In [ ]:
%python
w_code = Window.partitionBy("code")

df_reglas = (
    df_bronze
    .withColumn("code", F.trim(F.col("code")))
    .withColumn("description", F.trim(F.col("description")))
    .withColumn("id", F.col("id").cast("int"))
    .withColumn("status", F.col("status").cast("int"))
    .withColumn("version", F.col("version").cast("int"))
    .withColumn("_dup_count", F.count("*").over(w_code))
    # Regla 1: code vacío
    .withColumn("_r1", F.when((F.col("code").isNull()) | (F.col("code") == ""), F.lit("code_vacio")))
    # Regla 2: code duplicado (solo si no está vacío, para no solaparse con la Regla 1)
    .withColumn(
        "_r2",
        F.when(
            (F.col("code").isNotNull()) & (F.col("code") != "") & (F.col("_dup_count") > 1),
            F.lit("code_duplicado"),
        ),
    )
    .withColumn("_motivo_rechazo", F.concat_ws("; ", F.col("_r1"), F.col("_r2")))
    .withColumn(
        "_motivo_rechazo",
        F.when(F.col("_motivo_rechazo") == "", F.lit(None)).otherwise(F.col("_motivo_rechazo")),
    )
)

---

## Paso 2: Separar aceptados (Silver) de rechazados (DQL) y persistir

In [ ]:
%python
columnas_auxiliares = ["_dup_count", "_r1", "_r2"]

df_silver = (
    df_reglas.filter(F.col("_motivo_rechazo").isNull())
    .drop(*columnas_auxiliares, "_motivo_rechazo", "_ingested_at", "_source")
    .withColumn("_processed_at", F.current_timestamp())
)

df_dql = (
    df_reglas.filter(F.col("_motivo_rechazo").isNotNull())
    .drop(*columnas_auxiliares)
    .withColumn("_fecha_rechazo", F.current_timestamp())
)

total = df_bronze.count()
aceptados = df_silver.count()
rechazados = df_dql.count()
print(f"Bronze: {total:,} | Silver (aceptados): {aceptados:,} | DQL (rechazados): {rechazados:,}")
assert total == aceptados + rechazados, "Se perdieron o duplicaron filas en la separación Silver/DQL"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_SILVER)
df_dql.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_DQL)

print(f"Escrito: {TABLA_SILVER}")
print(f"Escrito: {TABLA_DQL}")

---

## Paso 3: Validación

In [ ]:
%sql
SELECT COUNT(*) AS total_silver FROM Datos_Empresas.silver.actividades_ciiu;

SELECT COUNT(*) AS total_dql FROM Datos_Empresas.dql.rechazados_ciiu;

In [ ]:
%sql
SELECT * FROM Datos_Empresas.silver.actividades_ciiu LIMIT 10;

---

## Siguiente paso

Continuar con `05_Silver_TRM.ipynb`, y luego `06_Gold_Empresas_Contratos.ipynb`.